In [ ]:
from pathlib import Path
import json
import numpy as np

NORMALIZED_SAVE_DIR = Path("/home/gella.saikrishna/code/Manual/saved_sleep_edf/normalized_psg")


def load_normalized_from_disk(load_path):
    saved = np.load(load_path, allow_pickle=True)

    metadata = json.loads(str(saved["metadata"].item()))
    hypnogram = saved["hypnogram"]

    if len(hypnogram) == 0:
        hypnogram = None

    participant = metadata["participant"]
    day = metadata["day"]
    key = (participant, day)

    psg_entry = {
        "participant": participant,
        "day": day,
        "psg_file": metadata["psg_file"],
        "channel_names": metadata["channel_names"],
        "sfreq": metadata["sfreq"],
        "normalized_data": saved["normalized_data"],
    }

    stats_entry = metadata.get("stats", {})

    hypno_entry = {
        "participant": participant,
        "day": day,
        "hypnogram_file": metadata.get("hypnogram_file"),
        "hypnogram": hypnogram,
    }

    return key, psg_entry, stats_entry, hypno_entry


normalized_psg_data = {}
psg_normalization_stats = {}
hypnogram_data = {}

saved_files = sorted(NORMALIZED_SAVE_DIR.glob("*.npz"))

print(f"Saved normalized PSG files found: {len(saved_files)}")

if len(saved_files) == 0:
    raise FileNotFoundError(f"No saved normalized files found in {NORMALIZED_SAVE_DIR}")

for load_path in saved_files:
    key, psg_entry, stats_entry, hypno_entry = load_normalized_from_disk(load_path)

    normalized_psg_data[key] = psg_entry
    psg_normalization_stats[key] = stats_entry
    hypnogram_data[key] = hypno_entry

print(f"Loaded normalized PSG files: {len(normalized_psg_data)}")
print(f"Loaded hypnograms: {len(hypnogram_data)}")

Saved normalized PSG files found: 153
Loaded normalized PSG files: 153
Loaded hypnograms: 153


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if torch.cuda.is_available():
    print("Visible GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))

Using device: cuda
Visible GPU count: 2
GPU name: NVIDIA RTX A4000


In [ ]:
from sklearn.model_selection import train_test_split

test_size = 0.2
random_state = 42

all_participants = sorted(set(key[0] for key in normalized_psg_data.keys()))

train_participants, test_participants = train_test_split(
    all_participants,
    test_size=test_size,
    random_state=random_state,
    shuffle=True,
)

train_participants = sorted(train_participants)
test_participants = sorted(test_participants)

print(f"Total participants: {len(all_participants)}")
print(f"Train participants: {len(train_participants)}")
print(f"Test participants: {len(test_participants)}")

print("\nTrain participants:")
print(train_participants)

print("\nTest participants:")
print(test_participants)

Total participants: 78
Train participants: 62
Test participants: 16

Train participants:
['SC401', 'SC402', 'SC403', 'SC405', 'SC406', 'SC407', 'SC408', 'SC409', 'SC411', 'SC413', 'SC414', 'SC415', 'SC416', 'SC417', 'SC419', 'SC420', 'SC421', 'SC423', 'SC424', 'SC425', 'SC426', 'SC427', 'SC428', 'SC429', 'SC431', 'SC432', 'SC435', 'SC436', 'SC437', 'SC438', 'SC440', 'SC441', 'SC442', 'SC443', 'SC444', 'SC446', 'SC447', 'SC448', 'SC449', 'SC451', 'SC452', 'SC453', 'SC454', 'SC456', 'SC457', 'SC458', 'SC459', 'SC460', 'SC461', 'SC463', 'SC464', 'SC465', 'SC466', 'SC470', 'SC471', 'SC472', 'SC473', 'SC474', 'SC476', 'SC477', 'SC480', 'SC481']

Test participants:
['SC400', 'SC404', 'SC410', 'SC412', 'SC418', 'SC422', 'SC430', 'SC433', 'SC434', 'SC445', 'SC450', 'SC455', 'SC462', 'SC467', 'SC475', 'SC482']


In [ ]:
new_training_size = 0.10

new_training_participants, _ = train_test_split(
    train_participants,
    train_size=new_training_size,
    random_state=random_state,
    shuffle=True,
)

new_training_participants = sorted(new_training_participants)

print(f"\nNew training participants: {len(new_training_participants)}")
print(new_training_participants)


New training participants: 6
['SC409', 'SC419', 'SC437', 'SC449', 'SC454', 'SC465']


In [ ]:
train_participants = new_training_participants
print("\nUpdated train participants:")
print(train_participants)


Updated train participants:
['SC409', 'SC419', 'SC437', 'SC449', 'SC454', 'SC465']


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.1):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=7,
            stride=stride,
            padding=3,
            bias=False,
        )
        self.bn1 = nn.BatchNorm1d(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=7,
            stride=1,
            padding=3,
            bias=False,
        )
        self.bn2 = nn.BatchNorm1d(out_channels)

        self.dropout = nn.Dropout(dropout)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm1d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)

        out = self.dropout(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + residual
        out = F.relu(out)

        return out


class SleepResNet1D(nn.Module):
    def __init__(self, n_channels=3, n_classes=6, dropout=0.2):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                n_channels,
                64,
                kernel_size=15,
                stride=2,
                padding=7,
                bias=False,
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )

        self.layer1 = nn.Sequential(
            ResidualBlock1D(64, 64, stride=1, dropout=dropout),
            ResidualBlock1D(64, 64, stride=1, dropout=dropout),
        )

        self.layer2 = nn.Sequential(
            ResidualBlock1D(64, 128, stride=2, dropout=dropout),
            ResidualBlock1D(128, 128, stride=1, dropout=dropout),
        )

        self.layer3 = nn.Sequential(
            ResidualBlock1D(128, 256, stride=2, dropout=dropout),
            ResidualBlock1D(256, 256, stride=1, dropout=dropout),
        )

        self.layer4 = nn.Sequential(
            ResidualBlock1D(256, 512, stride=2, dropout=dropout),
            ResidualBlock1D(512, 512, stride=1, dropout=dropout),
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, n_classes),
        )

    def forward(self, x):
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.global_pool(x)
        x = self.classifier(x)

        return x

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

2.6.0+cu124
True
12.4
NVIDIA RTX A4000


In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

selected_channels = ["EEG Fpz-Cz", "EEG Pz-Oz", "EMG submental"]
epoch_duration = 30


def get_channel_index(channel_names, target_name):
    target_lower = target_name.lower()

    for idx, ch in enumerate(channel_names):
        if ch.lower() == target_lower:
            return idx

    for idx, ch in enumerate(channel_names):
        if target_lower in ch.lower():
            return idx

    return None


def make_epochs_from_psg_entry(psg_entry, hypno_entry, selected_channels, epoch_duration=30):
    normalized_data = psg_entry["normalized_data"]
    channel_names = psg_entry["channel_names"]
    sfreq = psg_entry["sfreq"]
    hypnogram = hypno_entry["hypnogram"]

    if hypnogram is None:
        return None, None

    channel_indices = []

    for channel_name in selected_channels:
        idx = get_channel_index(channel_names, channel_name)

        if idx is None:
            print(f"Missing channel {channel_name} in {psg_entry['psg_file']}")
            return None, None

        channel_indices.append(idx)

    selected_data = normalized_data[channel_indices]

    samples_per_epoch = int(sfreq * epoch_duration)
    n_signal_epochs = selected_data.shape[1] // samples_per_epoch
    n_hypno_epochs = len(hypnogram)

    n_epochs = min(n_signal_epochs, n_hypno_epochs)

    selected_data = selected_data[:, :n_epochs * samples_per_epoch]

    selected_data = selected_data.reshape(
        len(selected_channels),
        n_epochs,
        samples_per_epoch,
    )

    X = np.transpose(selected_data, (1, 0, 2))
    y = np.array(hypnogram[:n_epochs], dtype=np.int64)

    valid_mask = y >= 0

    X = X[valid_mask]
    y = y[valid_mask]

    return X, y


X_train_list = []
y_train_list = []
X_test_list = []
y_test_list = []

for key, psg_entry in normalized_psg_data.items():
    participant = key[0]

    if key not in hypnogram_data:
        continue

    hypno_entry = hypnogram_data[key]

    X, y = make_epochs_from_psg_entry(
        psg_entry=psg_entry,
        hypno_entry=hypno_entry,
        selected_channels=selected_channels,
        epoch_duration=epoch_duration,
    )

    if X is None or y is None:
        continue

    if participant in train_participants:
        X_train_list.append(X)
        y_train_list.append(y)

    elif participant in test_participants:
        X_test_list.append(X)
        y_test_list.append(y)

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nTrain labels:", np.unique(y_train, return_counts=True))
print("Test labels:", np.unique(y_test, return_counts=True))

X_train shape: (31886, 3, 3000)
y_train shape: (31886,)
X_test shape: (87391, 3, 3000)
y_test shape: (87391,)

Train labels: (array([0, 1, 2, 3, 4, 5]), array([20422,  1823,  6487,   719,   195,  2240]))
Test labels: (array([0, 1, 2, 3, 4, 5]), array([60213,  4551, 14679,  1503,  1213,  5232]))


In [ ]:
# ============================================================================
# DATASET SIZE AND STATISTICS ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("DATASET SIZE AND STATISTICS")
print("="*80)

# Basic sizes
print(f"\n📊 BASIC SHAPES:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")

# Calculate total samples
n_train_samples = X_train.shape[0]
n_test_samples = X_test.shape[0]
total_samples = n_train_samples + n_test_samples

print(f"\n📈 SAMPLE COUNTS:")
print(f"  Training samples: {n_train_samples:,} ({n_train_samples/total_samples*100:.1f}%)")
print(f"  Test samples: {n_test_samples:,} ({n_test_samples/total_samples*100:.1f}%)")
print(f"  Total samples: {total_samples:,}")

# Channel and sequence info
n_channels = X_train.shape[1]
sequence_length = X_train.shape[2]

print(f"\n🔧 DATA DIMENSIONS:")
print(f"  Number of channels: {n_channels}")
print(f"  Channel names: {selected_channels}")
print(f"  Sequence length (30-sec epochs): {sequence_length} samples")
print(f"  Sampling frequency: {sequence_length / 30:.1f} Hz")

# Memory usage
memory_train_mb = X_train.nbytes / (1024 * 1024)
memory_test_mb = X_test.nbytes / (1024 * 1024)
memory_total_mb = memory_train_mb + memory_test_mb

print(f"\n💾 MEMORY USAGE:")
print(f"  X_train: {memory_train_mb:.2f} MB")
print(f"  X_test: {memory_test_mb:.2f} MB")
print(f"  Total: {memory_total_mb:.2f} MB")
print(f"  Approximate GPU memory needed: {memory_total_mb * 2:.2f} MB (with gradients)")

# Label distribution
train_unique, train_counts = np.unique(y_train, return_counts=True)
test_unique, test_counts = np.unique(y_test, return_counts=True)

# Sleep stage mapping
sleep_stages = {
    0: "Wake (W)",
    1: "NREM Stage 1 (N1)",
    2: "NREM Stage 2 (N2)",
    3: "NREM Stage 3 (N3)",
    4: "NREM Stage 4 (N4)",
    5: "REM (R)"
}

print(f"\n🏷️ LABEL DISTRIBUTION - TRAINING SET:")
print(f"  {'Stage':<20} {'Count':>10} {'Percentage':>12}")
print(f"  {'-'*42}")
for stage, count in zip(train_unique, train_counts):
    stage_name = sleep_stages.get(stage, f"Unknown ({stage})")
    percentage = count / n_train_samples * 100
    print(f"  {stage_name:<20} {count:>10,} {percentage:>11.2f}%")

print(f"\n🏷️ LABEL DISTRIBUTION - TEST SET:")
print(f"  {'Stage':<20} {'Count':>10} {'Percentage':>12}")
print(f"  {'-'*42}")
for stage, count in zip(test_unique, test_counts):
    stage_name = sleep_stages.get(stage, f"Unknown ({stage})")
    percentage = count / n_test_samples * 100
    print(f"  {stage_name:<20} {count:>10,} {percentage:>11.2f}%")

# Class imbalance check
print(f"\n⚖️ CLASS IMBALANCE RATIO (Train set):")
min_count = train_counts.min()
max_count = train_counts.max()
print(f"  Min class samples: {min_count:,}")
print(f"  Max class samples: {max_count:,}")
print(f"  Imbalance ratio: {max_count/min_count:.2f}:1")

# Participants info
print(f"\n👥 PARTICIPANT STATISTICS:")
print(f"  Training participants: {len(train_participants)}")
print(f"  Test participants: {len(test_participants)}")
print(f"  Total participants: {len(train_participants) + len(test_participants)}")

# Samples per participant (approximate)
samples_per_participant_train = n_train_samples / len(train_participants)
samples_per_participant_test = n_test_samples / len(test_participants)

print(f"  Avg samples/participant (train): {samples_per_participant_train:.1f}")
print(f"  Avg samples/participant (test): {samples_per_participant_test:.1f}")

# Data type check
print(f"\n🔍 DATA TYPE CHECK:")
print(f"  X_train dtype: {X_train.dtype}")
print(f"  y_train dtype: {y_train.dtype}")
print(f"  X_train value range: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"  y_train value range: [{y_train.min()}, {y_train.max()}]")

# Validation check
print(f"\n✅ VALIDATION CHECK:")
if len(train_unique) == 6:
    print("  ✓ All 6 sleep stages present in training set")
else:
    missing = set(range(6)) - set(train_unique)
    print(f"  ⚠️ Warning: Missing stages {missing} in training set")

if y_train.min() >= 0 and y_train.max() <= 5:
    print("  ✓ Labels are in valid range (0-5)")
else:
    print(f"  ⚠️ Warning: Labels outside 0-5 range detected")

print("="*80)


DATASET SIZE AND STATISTICS

📊 BASIC SHAPES:
  X_train shape: (31886, 3, 3000)
  y_train shape: (31886,)
  X_test shape: (87391, 3, 3000)
  y_test shape: (87391,)

📈 SAMPLE COUNTS:
  Training samples: 31,886 (26.7%)
  Test samples: 87,391 (73.3%)
  Total samples: 119,277

🔧 DATA DIMENSIONS:
  Number of channels: 3
  Channel names: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EMG submental']
  Sequence length (30-sec epochs): 3000 samples
  Sampling frequency: 100.0 Hz

💾 MEMORY USAGE:
  X_train: 1094.72 MB
  X_test: 3000.33 MB
  Total: 4095.05 MB
  Approximate GPU memory needed: 8190.10 MB (with gradients)

🏷️ LABEL DISTRIBUTION - TRAINING SET:
  Stage                     Count   Percentage
  ------------------------------------------
  Wake (W)                 20,422       64.05%
  NREM Stage 1 (N1)         1,823        5.72%
  NREM Stage 2 (N2)         6,487       20.34%
  NREM Stage 3 (N3)           719        2.25%
  NREM Stage 4 (N4)           195        0.61%
  REM (R)                   2,240  

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CHECKPOINT_DIR = Path("/home/gella.saikrishna/code/Manual/saved_sleep_edf/model_checkpoints_10")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = CHECKPOINT_DIR / "latest_checkpoint.pt"
best_checkpoint_path = CHECKPOINT_DIR / "best_checkpoint.pt"

batch_size = 64
num_epochs = 30
learning_rate = 1e-3
weight_decay = 1e-4

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = SleepResNet1D(n_channels=3, n_classes=6, dropout=0.2).to(device)

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

Using device: cuda


In [ ]:
from pathlib import Path
import torch
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CHECKPOINT_DIR = Path("/home/gella.saikrishna/code/Manual/saved_sleep_edf/model_checkpoints_10")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = CHECKPOINT_DIR / "latest_checkpoint.pt"
best_checkpoint_path = CHECKPOINT_DIR / "best_checkpoint.pt"

batch_size = 64
num_epochs = 30
learning_rate = 1e-3
weight_decay = 1e-4

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
)

model = SleepResNet1D(
    n_channels=3,
    n_classes=6,
    dropout=0.2,
).to(device)

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)


def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


def save_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    train_loss,
    test_loss,
    test_accuracy,
    best_test_accuracy,
):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "best_test_accuracy": best_test_accuracy,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
    }

    temp_path = checkpoint_path.with_name(checkpoint_path.stem + "_tmp.pt")
    torch.save(checkpoint, temp_path)
    temp_path.replace(checkpoint_path)


def load_checkpoint(checkpoint_path, model, optimizer, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_test_accuracy = checkpoint.get("best_test_accuracy", 0.0)

    return start_epoch, best_test_accuracy


start_epoch = 0
best_test_accuracy = 0.0

if latest_checkpoint_path.exists():
    start_epoch, best_test_accuracy = load_checkpoint(
        checkpoint_path=latest_checkpoint_path,
        model=model,
        optimizer=optimizer,
        device=device,
    )

    print(f"Resumed from checkpoint: {latest_checkpoint_path}")
    print(f"Starting from epoch: {start_epoch + 1}")
    print(f"Best previous test accuracy: {best_test_accuracy * 100:.2f}%")
else:
    print("No checkpoint found. Starting training from epoch 1.")


for epoch in range(start_epoch, num_epochs):
    model.train()

    running_loss = 0.0
    total_train_samples = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)
        total_train_samples += X_batch.size(0)

    train_loss = running_loss / total_train_samples

    test_loss, test_accuracy = evaluate_model(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
    )

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Accuracy: {test_accuracy * 100:.2f}%"
    )

    save_checkpoint(
        checkpoint_path=latest_checkpoint_path,
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        train_loss=train_loss,
        test_loss=test_loss,
        test_accuracy=test_accuracy,
        best_test_accuracy=best_test_accuracy,
    )

    if test_accuracy > best_test_accuracy:
        best_test_accuracy = test_accuracy

        save_checkpoint(
            checkpoint_path=best_checkpoint_path,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            train_loss=train_loss,
            test_loss=test_loss,
            test_accuracy=test_accuracy,
            best_test_accuracy=best_test_accuracy,
        )

        print(f"Saved best checkpoint: {best_test_accuracy * 100:.2f}%")

print("\nTraining complete.")
print(f"Latest checkpoint: {latest_checkpoint_path}")
print(f"Best checkpoint: {best_checkpoint_path}")

Using device: cuda
GPU: NVIDIA RTX A4000
No checkpoint found. Starting training from epoch 1.
Epoch [1/30] Train Loss: 0.4151 | Test Loss: 0.3638 | Test Accuracy: 86.63%
Saved best checkpoint: 86.63%
Epoch [2/30] Train Loss: 0.2975 | Test Loss: 0.3195 | Test Accuracy: 87.70%
Saved best checkpoint: 87.70%
Epoch [3/30] Train Loss: 0.2754 | Test Loss: 0.3070 | Test Accuracy: 88.66%
Saved best checkpoint: 88.66%
Epoch [4/30] Train Loss: 0.2587 | Test Loss: 0.3362 | Test Accuracy: 87.27%
Epoch [5/30] Train Loss: 0.2488 | Test Loss: 0.3099 | Test Accuracy: 88.81%
Saved best checkpoint: 88.81%
Epoch [6/30] Train Loss: 0.2426 | Test Loss: 0.3005 | Test Accuracy: 89.25%
Saved best checkpoint: 89.25%
Epoch [7/30] Train Loss: 0.2344 | Test Loss: 0.2909 | Test Accuracy: 89.06%
Epoch [8/30] Train Loss: 0.2313 | Test Loss: 0.3041 | Test Accuracy: 88.96%
Epoch [9/30] Train Loss: 0.2263 | Test Loss: 0.3084 | Test Accuracy: 88.49%
Epoch [10/30] Train Loss: 0.2231 | Test Loss: 0.3027 | Test Accuracy: 88